# A full business solution

## Now we will take our project from Day 1 to the next level

### BUSINESS CHALLENGE:

Create a product that builds a Brochure for a company to be used for prospective clients, investors and potential recruits.

We will be provided a company name and their primary website.

See the end of this notebook for examples of real-world business applications.

And remember: I'm always available if you have problems or ideas! Please do reach out.

In [39]:
# imports
# If these fail, please check you're running from an 'activated' environment with (llms) in the command prompt

import os
import json
from dotenv import load_dotenv
from IPython.display import Markdown, display, update_display
from scraper import fetch_website_links, fetch_website_contents
from openai import OpenAI

In [48]:
# Initialize and constants

load_dotenv(override=True)
api_key = os.getenv('GOOGLE_API_KEY')

if api_key and api_key.startswith('sk-proj-') and len(api_key)>10:
    print("API key looks good so far")
else:
    print("There might be a problem with your API key? Please visit the troubleshooting notebook!")
    
MODEL = 'gemini-3.5-flash-lite'
GEMINI_BASE_URL= "https://generativelanguage.googleapis.com/v1beta/openai/"
openai = OpenAI(base_url=GEMINI_BASE_URL, api_key=api_key)

There might be a problem with your API key? Please visit the troubleshooting notebook!


In [42]:
links = fetch_website_links("https://edwarddonner.com")
links

['#wp--skip-link--target',
 'https://edwarddonner.com/avatar/',
 'https://edwarddonner.com/curriculum/',
 'https://edwarddonner.com/proficient/',
 'https://edwarddonner.com/connect-four/',
 'https://edwarddonner.com/outsmart/',
 'https://edwarddonner.com/about-me-and-about-nebula/',
 'https://edwarddonner.com/posts/',
 'https://edwarddonner.com/',
 'https://news.ycombinator.com',
 'https://nebula.io/?utm_source=ed&utm_medium=referral',
 'https://www.prnewswire.com/news-releases/wynden-stark-group-acquires-nyc-venture-backed-tech-startup-untapt-301269512.html',
 'https://edwarddonner.com/curriculum/',
 'https://edwarddonner.com/avatar/',
 'https://edwarddonner.com/2026/10/14/ai-fde-the-forward-deployed-engineer-agentic-ai-architect/',
 'https://edwarddonner.com/2026/10/14/ai-fde-the-forward-deployed-engineer-agentic-ai-architect/',
 'https://edwarddonner.com/2026/02/17/ai-coder-vibe-coder-to-agentic-engineer/',
 'https://edwarddonner.com/2026/02/17/ai-coder-vibe-coder-to-agentic-enginee

## First step: Have GPT-6 Luna figure out which links are relevant

### Use a call to GPT-6 Luna to read the links on a webpage, and respond in structured JSON.  
It should decide which links are relevant, and replace relative links such as "/about" with "https://company.com/about".  
We will use "one shot prompting" in which we provide an example of how it should respond in the prompt.

This is an excellent use case for an LLM, because it requires nuanced understanding. Imagine trying to code this without LLMs by parsing and analyzing the webpage - it would be very hard!

Sidenote: there is a more advanced technique called "Structured Outputs" in which we require the model to respond according to a spec. We cover this technique in Week 8 during our autonomous Agentic AI project.

In [43]:
link_system_prompt = """
You are provided with a list of links found on a webpage.
You are able to decide which of the links would be most relevant to include in a brochure about the company,
such as links to an About page, or a Company page, or Careers/Jobs pages.
You should respond in JSON as in this example:

{
    "links": [
        {"type": "about page", "url": "https://full.url/goes/here/about"},
        {"type": "careers page", "url": "https://another.full.url/careers"}
    ]
}
"""

In [44]:
def get_links_user_prompt(url):
    user_prompt = f"""
Here is the list of links on the website {url} -
Please decide which of these are relevant web links for a brochure about the company, 
respond with the full https URL in JSON format.
Do not include Terms of Service, Privacy, email links.

Links (some might be relative links):

"""
    links = fetch_website_links(url)
    user_prompt += "\n".join(links)
    return user_prompt

In [45]:
print(get_links_user_prompt("https://edwarddonner.com"))


Here is the list of links on the website https://edwarddonner.com -
Please decide which of these are relevant web links for a brochure about the company, 
respond with the full https URL in JSON format.
Do not include Terms of Service, Privacy, email links.

Links (some might be relative links):

#wp--skip-link--target
https://edwarddonner.com/avatar/
https://edwarddonner.com/curriculum/
https://edwarddonner.com/proficient/
https://edwarddonner.com/connect-four/
https://edwarddonner.com/outsmart/
https://edwarddonner.com/about-me-and-about-nebula/
https://edwarddonner.com/posts/
https://edwarddonner.com/
https://news.ycombinator.com
https://nebula.io/?utm_source=ed&utm_medium=referral
https://www.prnewswire.com/news-releases/wynden-stark-group-acquires-nyc-venture-backed-tech-startup-untapt-301269512.html
https://edwarddonner.com/curriculum/
https://edwarddonner.com/avatar/
https://edwarddonner.com/2026/10/14/ai-fde-the-forward-deployed-engineer-agentic-ai-architect/
https://edwarddon

In [54]:
def select_relevant_links(url):
    response = openai.chat.completions.create(
        model=MODEL,
        
        messages=[
            {"role": "system", "content": link_system_prompt},
            {"role": "user", "content": get_links_user_prompt(url)}
        ],
        response_format={"type": "json_object"}
    )
    result = response.choices[0].message.content
    links = json.loads(result)
    return links
    

In [55]:
select_relevant_links("https://edwarddonner.com")

{'links': [{'type': 'about page',
   'url': 'https://edwarddonner.com/about-me-and-about-nebula/'},
  {'type': 'portfolio page', 'url': 'https://edwarddonner.com/curriculum/'},
  {'type': 'blog page', 'url': 'https://edwarddonner.com/posts/'}]}

In [58]:
def select_relevant_links(url):
    print(f"Selecting relevant links for {url} by calling {MODEL}")
    response = openai.chat.completions.create(
        model=MODEL,
        
        messages=[
            {"role": "system", "content": link_system_prompt},
            {"role": "user", "content": get_links_user_prompt(url)}
        ],
        response_format={"type": "json_object"}
    )
    result = response.choices[0].message.content
    links = json.loads(result)
    print(f"Found {len(links['links'])} relevant links")
    return links

In [59]:
select_relevant_links("https://edwarddonner.com")

Selecting relevant links for https://edwarddonner.com by calling gemini-3.5-flash-lite
Found 3 relevant links


{'links': [{'type': 'about page',
   'url': 'https://edwarddonner.com/about-me-and-about-nebula/'},
  {'type': 'curriculum page', 'url': 'https://edwarddonner.com/curriculum/'},
  {'type': 'blog posts page', 'url': 'https://edwarddonner.com/posts/'}]}

In [60]:
select_relevant_links("https://huggingface.co")

Selecting relevant links for https://huggingface.co by calling gemini-3.5-flash-lite
Found 8 relevant links


{'links': [{'type': 'company page', 'url': 'https://huggingface.co'},
  {'type': 'blog', 'url': 'https://huggingface.co/blog'},
  {'type': 'pricing page', 'url': 'https://huggingface.co/pricing'},
  {'type': 'enterprise page', 'url': 'https://huggingface.co/enterprise'},
  {'type': 'brand guidelines', 'url': 'https://huggingface.co/brand'},
  {'type': 'careers page', 'url': 'https://apply.workable.com/huggingface/'},
  {'type': 'github page', 'url': 'https://github.com/huggingface'},
  {'type': 'linkedin page',
   'url': 'https://www.linkedin.com/company/huggingface/'}]}

## Second step: make the brochure!

Assemble all the details into another prompt to GPT-6 Luna

In [61]:
def fetch_page_and_all_relevant_links(url):
    contents = fetch_website_contents(url)
    relevant_links = select_relevant_links(url)
    result = f"## Landing Page:\n\n{contents}\n## Relevant Links:\n"
    for link in relevant_links['links']:
        result += f"\n\n### Link: {link['type']}\n"
        result += fetch_website_contents(link["url"])
    return result

In [63]:
print(fetch_page_and_all_relevant_links("https://huggingface.co"))

Selecting relevant links for https://huggingface.co by calling gemini-3.5-flash-lite
Found 9 relevant links
## Landing Page:

Hugging Face – The AI community building the future.

Hugging Face
Models
Datasets
Spaces
Buckets
Docs
Enterprise
Pricing
Website
Tasks
HuggingChat
Collections
Languages
Organizations
Community
Blog
Posts
Daily Papers
Hardware
Learn
Discord
Forum
GitHub
Solutions
Team & Enterprise
Hugging Face PRO
Enterprise Support
Inference Providers
Inference Endpoints
Storage Buckets
Log In
Sign Up
The AI community building the future.
The platform where the machine learning community collaborates on models, datasets, and applications.
Explore AI Apps
or
Browse 2M+ models
Trending on
this week
Models
LiquidAI/LFM3-1.7B
Updated
5 days ago
•
42.4k
•
842
deepseek-ai/DeepSeek-V4.1-Flash-1017
Updated
16 days ago
•
177k
•
4.65k
nvidia/NVIDIA-Nemotron-3.5-Transcribe-30B-A4B
Updated
4 days ago
•
32.4k
•
659
mistralai/Mistral-Large-4-1T-A52B
Updated
about 1 hour ago
•
5.5k
•
1.48k
ba

In [77]:
#brochure_system_prompt = """
#
# Or uncomment the lines below for a more humorous brochure - this demonstrates how easy it is to incorporate 'tone':

brochure_system_prompt = """
You are an assistant that analyzes the contents of several relevant pages from a company website
and creates a short, humorous, entertaining, witty brochure about the company for prospective customers, investors and recruits.
Respond in markdown without code blocks.
Include details of company culture, customers and careers/jobs if you have the information.
"""


In [65]:
def get_brochure_user_prompt(company_name, url):
    user_prompt = f"""
You are looking at a company called: {company_name}
Here are the contents of its landing page and other relevant pages;
use this information to build a short brochure of the company in markdown without code blocks.\n\n
"""
    user_prompt += fetch_page_and_all_relevant_links(url)
    user_prompt = user_prompt[:5_000] # Truncate if more than 5,000 characters
    return user_prompt

In [66]:
get_brochure_user_prompt("HuggingFace", "https://huggingface.co")

Selecting relevant links for https://huggingface.co by calling gemini-3.5-flash-lite
Found 9 relevant links


'\nYou are looking at a company called: HuggingFace\nHere are the contents of its landing page and other relevant pages;\nuse this information to build a short brochure of the company in markdown without code blocks.\n\n\n## Landing Page:\n\nHugging Face – The AI community building the future.\n\nHugging Face\nModels\nDatasets\nSpaces\nBuckets\nDocs\nEnterprise\nPricing\nWebsite\nTasks\nHuggingChat\nCollections\nLanguages\nOrganizations\nCommunity\nBlog\nPosts\nDaily Papers\nHardware\nLearn\nDiscord\nForum\nGitHub\nSolutions\nTeam & Enterprise\nHugging Face PRO\nEnterprise Support\nInference Providers\nInference Endpoints\nStorage Buckets\nLog In\nSign Up\nThe AI community building the future.\nThe platform where the machine learning community collaborates on models, datasets, and applications.\nExplore AI Apps\nor\nBrowse 2M+ models\nTrending on\nthis week\nModels\nLiquidAI/LFM3-1.7B\nUpdated\n5 days ago\n•\n42.4k\n•\n842\ndeepseek-ai/DeepSeek-V4.1-Flash-1017\nUpdated\n16 days ago\n•\

In [71]:
def create_brochure(company_name, url):
    response = openai.chat.completions.create(
        model="gemini-3.5-flash-lite",
        
        messages=[
            {"role": "system", "content": brochure_system_prompt},
            {"role": "user", "content": get_brochure_user_prompt(company_name, url)}
        ],
    )
    result = response.choices[0].message.content
    display(Markdown(result))

In [72]:
create_brochure("HuggingFace", "https://huggingface.co")

Selecting relevant links for https://huggingface.co by calling gemini-3.5-flash-lite
Found 8 relevant links


# Hugging Face: The AI Community Building the Future

Welcome to Hugging Face, the home and premier collaboration platform for machine learning. We are the central hub where the global AI community comes together to create, discover, build, and deploy the next generation of artificial intelligence. 

## What We Offer

Hugging Face empowers developers, researchers, and organizations by providing a robust ecosystem for machine learning assets:

- **Over 2 Million Models:** Explore and leverage a massive repository of cutting-edge open-source models, ranging from advanced language models and vision systems to audio and multimodal architectures.
- **Over 500,000 Datasets:** Access diverse data collections spanning text, images, 3D assets, cybersecurity, finance, and beyond to train and fine-tune your models.
- **Interactive Spaces:** Discover, test, and deploy over 1 million AI applications and interactive demos powered by state-of-the-art hardware and agent frameworks.
- **Enterprise Solutions:** Scale your AI initiatives securely with Hugging Face PRO, Enterprise Support, Inference Endpoints, and dedicated storage buckets tailored for team collaboration.

## Company Culture & Community

At our core, Hugging Face is driven by a vibrant, global community. Our culture thrives on open-source collaboration, shared research, and ethical AI development. Whether you are discussing the latest breakthroughs on our Forum and Discord, exploring Daily Papers, or collaborating through GitHub, you will find a welcoming environment dedicated to pushing the boundaries of technology together. We cover diverse domains including Natural Language Processing (NLP), Computer Vision (CV), Audio, Reinforcement Learning (RL), and Diffusion models.

## Customers & Partners

Hugging Face partners with industry leaders and innovators—including tech giants like NVIDIA and top-tier AI labs—to provide world-class infrastructure, inference providers, and intelligent tools. From individual developers building creative side projects to enterprise organizations scaling production machine learning workloads, our platform serves builders across all industries.

## Join Us

Are you passionate about shaping the future of AI? We are always looking for curious, driven individuals to join our community and team. Dive into our platform, contribute to open-source models, or check out our latest blog posts and career opportunities to see how you can help us build the future of machine learning.

## Finally - a minor improvement

With a small adjustment, we can change this so that the results stream back from OpenAI,
with the familiar typewriter animation

In [75]:
def stream_brochure(company_name, url):
    stream = openai.chat.completions.create(
        model="gemini-3.1-flash-lite",
        
        messages=[
            {"role": "system", "content": brochure_system_prompt},
            {"role": "user", "content": get_brochure_user_prompt(company_name, url)}
          ],
        stream=True
    )    
    response = ""
    display_handle = display(Markdown(""), display_id=True)
    for chunk in stream:
        response += chunk.choices[0].delta.content or ''
        update_display(Markdown(response), display_id=display_handle.display_id)

In [76]:
stream_brochure("HuggingFace", "https://huggingface.co")

Selecting relevant links for https://huggingface.co by calling gemini-3.5-flash-lite
Found 10 relevant links


# Hugging Face: The AI Community Building the Future

## Who We Are
Hugging Face is the central hub for the machine learning community. We provide the essential infrastructure that enables researchers, developers, and organizations to create, discover, and collaborate on artificial intelligence. Whether you are working with Large Language Models (LLMs), computer vision, audio, or reinforcement learning, Hugging Face serves as the collaborative platform where the future of AI is being built.

## Our Platform
We provide a comprehensive ecosystem designed for the modern AI workflow:
* **Models:** Explore over 2 million machine learning models covering every major architecture and domain.
* **Datasets:** Access over 500,000 datasets to fuel your training and research needs.
* **Spaces:** Showcase and deploy your AI applications—from video generation to interactive 3D simulations—on our cloud infrastructure.
* **Enterprise Solutions:** We support professional teams with inference endpoints, dedicated storage, and enterprise-grade support, ensuring that companies can scale their AI production securely and efficiently.

## Our Community & Culture
At the heart of Hugging Face is a commitment to open-source collaboration. We believe that AI progress accelerates when the community shares knowledge openly. Our culture is defined by:
* **Radical Collaboration:** We bridge the gap between individual researchers, hobbyists, and global tech leaders like NVIDIA and Mistral.
* **Continuous Learning:** Through our daily papers, educational documentation, and vibrant Discord and Forum communities, we foster an environment of constant discovery.
* **Open Science:** We lead the conversation on AI ethics, research, and technical breakthroughs, ensuring that the development of AI remains transparent and accessible.

## Why Join Us?
Hugging Face is more than a platform; it is the industry standard for machine learning infrastructure. By joining our team or utilizing our enterprise tools, you are positioning yourself at the epicenter of the AI revolution. 

* **For Customers:** Leverage our robust inference providers and scalable endpoints to bring your AI models from experimentation to production.
* **For Investors:** We are the essential "infrastructure layer" of the AI stack, hosting the most important assets in the machine learning ecosystem.
* **For Talent:** If you are passionate about building the tools that will shape the next decade of technology, we offer the chance to work on the most significant challenges in machine learning while contributing to the global open-source community.

**Join the movement. Start building with us today at HuggingFace.co.**

In [78]:
# Try changing the system prompt to the humorous version when you make the Brochure for Hugging Face:

stream_brochure("HuggingFace", "https://huggingface.co")

Selecting relevant links for https://huggingface.co by calling gemini-3.5-flash-lite
Found 9 relevant links


# Welcome to Hugging Face: The Internet’s Favorite Emoji (and AI Powerhouse)

### Seriously, Who Are We?
If you’ve ever wondered who is responsible for the sheer volume of AI models flooding your feed, look no further. We are Hugging Face. We aren’t just a company; we’re the global town square for the machine learning community. Think of us as the "GitHub for AI," but with significantly more emojis and way fewer existential crises.

### Why Do People Hang Out Here?
We’ve built the ultimate sandbox for the curious and the brilliant. Whether you are an AI researcher, a developer, or someone who just really likes making computers do funny things, we’ve got the goods:

*   **2 Million+ Models:** From linguistic geniuses to image-generating wizards.
*   **500,000+ Datasets:** Because an AI without data is just a very expensive paperweight.
*   **Spaces:** A place to host your AI apps, show off your demos, and let the world play with your creations.

### The Culture: Chaos, Collaboration, and Code
Our culture is built on the wild, open-source spirit. We believe that if you want to go fast, go alone; if you want to change the world (and maybe accidentally build a robot that writes better poetry than you), you should do it on Hugging Face. We trade stuffy boardrooms for Discord channels, and we treat documentation like a religion. If it isn't documented, did it even happen?

### Who Uses Hugging Face?
Everyone from the biggest tech giants to the lone dev working in their basement at 3 AM. If you’re pushing the boundaries of NLP, Audio, Computer Vision, or just trying to swap a face on a photo to look like a medieval peasant, you’re likely using our tools. We’re the infrastructure under the hood of the AI revolution.

### Careers: Do You Hug?
Are you obsessed with machine learning? Does the idea of democratizing AI make you feel all warm and fuzzy inside? We’re looking for the sharpest minds to help us keep the servers humming and the community thriving. 

Joining Hugging Face means:
1.  **Working at the cutting edge:** We’re not just following trends; we’re setting them.
2.  **Global Impact:** Your work won't just sit in a private repository—it will be used by millions.
3.  **The Emoji Cred:** You get to be part of the brand that made " 🤗 " the most intimidatingly powerful symbol in tech.

### Join the Revolution
Whether you’re an investor looking for the heart of the AI ecosystem, a developer hunting for the next big model, or a recruit looking for your next adventure—come pull up a chair. 

**Hugging Face: We’re building the future, one model at a time. And yes, we really like emojis.** 

*Check us out at HuggingFace.co—unless you’re afraid of the future. (Don’t be. It’s actually quite fun.)*

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/business.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#181;">Business applications</h2>
            <span style="color:#181;">In this exercise we extended the Day 1 code to make multiple LLM calls, and generate a document.

This is perhaps the first example of Agentic AI design patterns, as we combined multiple calls to LLMs. This will feature more in Week 2, and then we will return to Agentic AI in a big way in Week 8 when we build a fully autonomous Agent solution.

Generating content in this way is one of the very most common Use Cases. As with summarization, this can be applied to any business vertical. Write marketing content, generate a product tutorial from a spec, create personalized email content, and so much more. Explore how you can apply content generation to your business, and try making yourself a proof-of-concept prototype. See what other students have done in the community-contributions folder -- so many valuable projects -- it's wild!</span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/important.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#900;">Before you move to Week 2 (which is tons of fun)</h2>
            <span style="color:#900;">Please see the week1 EXERCISE notebook for your challenge for the end of week 1. This will give you some essential practice working with Frontier APIs, and prepare you well for Week 2.</span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/resources.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#f71;">A reminder on 3 useful resources</h2>
            <span style="color:#f71;">1. The resources for the course are available <a href="https://edwarddonner.com/2024/11/13/llm-engineering-resources/">here.</a><br/>
            2. I'm on LinkedIn <a href="https://www.linkedin.com/in/eddonner/">here</a> and I love connecting with people taking the course!<br/>
            3. I'm trying out X/Twitter and I'm at <a href="https://x.com/edwarddonner">@edwarddonner<a> and hoping people will teach me how it's done..  
            </span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/thankyou.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#090;">Finally! I have a special request for you</h2>
            <span style="color:#090;">
                My editor tells me that it makes a MASSIVE difference when students rate this course on Udemy - it's one of the main ways that Udemy decides whether to show it to others. If you're able to take a minute to rate this, I'd be so very grateful! And regardless - always please reach out to me at ed@edwarddonner.com if I can help at any point.
            </span>
        </td>
    </tr>
</table>